# Selenium을 이용한 기상청 날씨 크롤링

In [1]:
%pip install -q selenium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: c:\Users\dandycode\.pyenv\pyenv-win\versions\3.11.7\python.exe -m pip install --upgrade pip


In [3]:
# 봇 처럼 여겨지지 않기 위해 주피터 노트북 ipynb 파일 생성
# 크롤링은 어떻게 사이트에서 사람이 하는 것처럼 보일까가 중요

# pip install selenium
from selenium import webdriver

driver = webdriver.Chrome()
# driver.set_window_size(1920, 1080)
driver.set_window_size(1280, 720)

# URL='https://www.naver.com/'
URL='https://www.weather.go.kr/w/weather/forecast/short-term.do'
driver.get(url=URL)

In [4]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [5]:
# --- 방법 1: 링크 텍스트 사용 (가장 간단하고 추천) ---
# "1시간 간격"이라는 텍스트를 가진 링크를 직접 찾습니다.
print("방법 1: 링크 텍스트로 클릭 시도...")
# WebDriverWait를 사용하여 요소가 클릭 가능할 때까지 최대 10초간 기다립니다.
one_hour_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.LINK_TEXT, "1시간 간격"))
)
one_hour_button.click()
print("'1시간 간격' 버튼 클릭 성공 (링크 텍스트 사용)")

방법 1: 링크 텍스트로 클릭 시도...
'1시간 간격' 버튼 클릭 성공 (링크 텍스트 사용)


In [6]:
# --- 방법 2: CSS 선택자 사용 ---
print("방법 2: CSS 선택자(클래스)로 클릭 시도...")
table_view_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "a.view-table"))
)
table_view_button.click()
print("'표 형태' 버튼 클릭 성공 (CSS 선택자 - 클래스 사용)")

방법 2: CSS 선택자(클래스)로 클릭 시도...
'표 형태' 버튼 클릭 성공 (CSS 선택자 - 클래스 사용)


In [7]:
from selenium.common.exceptions import NoSuchElementException
import re # 정규표현식 사용 (데이터 정제용)

In [8]:
# --- 데이터 저장을 위한 빈 리스트 초기화 ---
times = []
weathers = []
temperatures = []
feels_like_temps = []
precip_amounts = []
precip_intensities = []
precip_probabilities = []
wind_directions = []
wind_speeds = []
humidities = []
heatwave_impacts = []

# --- 데이터 추출 로직 ---
try:
    # 데이터 항목들을 포함하는 부모 div 찾기
    # daily_div = driver.find_element(By.CSS_SELECTOR, "div.daily")
    # item_wrap = daily_div.find_element(By.CSS_SELECTOR, "div.item-wrap")
    # 위 코드를 > 를 이용해 한줄로 작성 가능
    item_wrap = driver.find_element(By.CSS_SELECTOR, "div.daily > div.item-wrap")

    # print(item_wrap.get_attribute('outerHTML')) # item_wrap 내용 확인

    # 각 시간대별 데이터 묶음 (ul 태그) 찾기
    item_list = item_wrap.find_elements(By.CSS_SELECTOR, "ul.item")

    print(f"총 {len(item_list)}개의 시간대 데이터를 찾았습니다.")

    # 각 시간대별로 반복 처리
    for item_ul in item_list:
        # 각 ul 내의 li 요소들을 리스트로 가져오기
        # IndexError를 방지하기 위해 li 개수를 먼저 확인하는 것이 더 안전할 수 있습니다.
        try:
             li_elements = item_ul.find_elements(By.TAG_NAME, "li")
             # 최소 필요한 li 개수(예: 10개) 확인 로직 추가 가능
             # if len(li_elements) < 10: continue # 또는 None 추가 후 다음 item으로
        except NoSuchElementException:
             print("경고: 현재 시간대(ul.item)에서 li 요소들을 찾을 수 없습니다. 건너<0xEB><0x8D>니다.")
             # 모든 리스트에 None 추가하고 다음 item_ul로 넘어감
             lists_to_update = [times, weathers, temperatures, feels_like_temps, precip_amounts, precip_intensities, precip_probabilities, wind_directions, wind_speeds, humidities, heatwave_impacts]
             for lst in lists_to_update:
                 lst.append(None)
             continue # 다음 시간대로

        # 각 항목 추출 및 정제 (clean_value 함수 로직 인라인)

        # 1. 시각
        cleaned_time = None
        try:
            time_text = li_elements[0].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            # 정제 로직 (기본 전처리)
            time_text = time_text.strip().replace('&nbsp;', '')
            if time_text and time_text != '-':
                cleaned_time = time_text # 시각은 특별한 숫자 변환 없음
        except (NoSuchElementException, IndexError) as e:
             print(f"시각 처리 오류: {e}") # 디버깅용 로그
        times.append(cleaned_time)


        # 2. 날씨
        cleaned_weather = None
        try:
            # 먼저 wic 클래스 시도
            try:
                 weather_text = li_elements[1].find_element(By.CSS_SELECTOR, "span.wic").text
            except NoSuchElementException:
                 # wic 없으면 다른 span 시도
                 weather_text = li_elements[1].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            
            # 정제 로직 (기본 전처리)
            weather_text = weather_text.strip().replace('&nbsp;', '')
            if weather_text and weather_text != '-':
                 cleaned_weather = weather_text # 날씨는 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
            print(f"날씨 처리 오류: {e}")
        weathers.append(cleaned_weather)


        # 3. 기온 
        # 참고: 원래 코드에서는 li_elements[2] (3번째 li)에서 추출했으나, 
        # 이전 논의에서 실제 기온은 4번째 li에서 가져오는 것이 맞다고 판단했습니다.
        # 만약 3번째 li의 텍스트에서 첫 숫자를 기온으로 사용하려면 아래 로직 사용
        cleaned_temp = None
        try:
             # 3번째 li의 전체 텍스트 (예: "16℃(16℃)") 에서 첫 숫자 추출
             temp_text_combined = li_elements[2].find_element(By.CSS_SELECTOR, "span.hid.feel").text
             temp_text_combined = temp_text_combined.strip().replace('&nbsp;', '')
             if temp_text_combined and temp_text_combined != '-':
                 match = re.search(r'-?\d+', temp_text_combined) # 첫 번째 숫자 검색
                 if match:
                     cleaned_temp = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"기온 처리 오류: {e}")
        temperatures.append(cleaned_temp)
        

        # 4. 기온 체감 (3번째 li의 span.chill 텍스트)
        cleaned_feels_like = None
        try:
            chill_text = li_elements[2].find_element(By.CSS_SELECTOR, "span.chill").text # 예: (16℃)
            chill_text = chill_text.strip().replace('&nbsp;', '')
            if chill_text and chill_text != '-':
                match = re.search(r'-?\d+', chill_text) # 괄호 안 숫자 검색
                if match:
                    cleaned_feels_like = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"체감기온 처리 오류: {e}")
        feels_like_temps.append(cleaned_feels_like)


        # 5. 강수량
        cleaned_precip_amount = None
        try:
            pcp_text = li_elements[4].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            pcp_text = pcp_text.strip().replace('&nbsp;', '')
            if pcp_text and pcp_text != '-':
                if '빗방울' in pcp_text:
                    cleaned_precip_amount = 0.0 # '빗방울'은 0.0으로 처리
                else:
                    match = re.search(r'\d+\.?\d*', pcp_text) # 소수점 포함 숫자 검색
                    if match:
                        cleaned_precip_amount = float(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"강수량 처리 오류: {e}")
        precip_amounts.append(cleaned_precip_amount)


        # 6. 강수강도
        cleaned_intensity = None
        try:
            intensity_element = li_elements[5]
            intensity_text = None
            # 먼저 span 찾기 시도
            try:
                intensity_text = intensity_element.find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            except NoSuchElementException:
                 # span 없으면 li 전체 텍스트에서 hid 제외
                 full_text = intensity_element.text
                 hidden_text = ""
                 try:
                     hidden_text = intensity_element.find_element(By.CSS_SELECTOR, "span.hid").text
                 except NoSuchElementException: pass
                 intensity_text = full_text.replace(hidden_text, "").strip()

            # 정제 로직 (기본 전처리)
            intensity_text = intensity_text.strip().replace('&nbsp;', '')
            if intensity_text and intensity_text != '-':
                cleaned_intensity = intensity_text # 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
             print(f"강수강도 처리 오류: {e}")
        precip_intensities.append(cleaned_intensity)


        # 7. 강수확률
        cleaned_prob = None
        try:
            prob_text = li_elements[6].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            prob_text = prob_text.strip().replace('&nbsp;', '')
            if prob_text and prob_text != '-':
                match = re.search(r'\d+', prob_text) # % 제거 후 숫자만
                if match:
                    cleaned_prob = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"강수확률 처리 오류: {e}")
        precip_probabilities.append(cleaned_prob)


        # 8. 바람 (방향, 속도 분리)
        cleaned_wind_dir = None
        cleaned_wind_spd = None
        try:
            wind_li = li_elements[7]
            # 바람 방향
            try:
                wind_dir_text = wind_li.find_element(By.CSS_SELECTOR, "span.wdic").text
                wind_dir_text = wind_dir_text.strip().replace('&nbsp;', '')
                if wind_dir_text and wind_dir_text != '-':
                     cleaned_wind_dir = wind_dir_text
            except NoSuchElementException: pass # 없으면 None 유지
            # 바람 속도
            try:
                wind_spd_text = wind_li.find_element(By.CSS_SELECTOR, "span.wspd:not(.qwsd)").text
                wind_spd_text = wind_spd_text.strip().replace('&nbsp;', '')
                if wind_spd_text and wind_spd_text != '-':
                    match = re.search(r'\d+', wind_spd_text) # m/s 제거 후 숫자만
                    if match:
                        cleaned_wind_spd = int(match.group(0))
            except NoSuchElementException: pass # 없으면 None 유지
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"바람 처리 오류: {e}")
        wind_directions.append(cleaned_wind_dir)
        wind_speeds.append(cleaned_wind_spd)


        # 9. 습도
        cleaned_humidity = None
        try:
            hum_text = li_elements[8].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            hum_text = hum_text.strip().replace('&nbsp;', '')
            if hum_text and hum_text != '-':
                match = re.search(r'\d+', hum_text) # % 제거 후 숫자만
                if match:
                    cleaned_humidity = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"습도 처리 오류: {e}")
        humidities.append(cleaned_humidity)


        # 10. 폭염 영향
        cleaned_heatwave = None
        try:
            heat_text = li_elements[9].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            heat_text = heat_text.strip().replace('&nbsp;', '')
            if heat_text and heat_text != '-':
                cleaned_heatwave = heat_text # 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
            print(f"폭염영향 처리 오류: {e}")
        heatwave_impacts.append(cleaned_heatwave)

    # --- 최종 결과 출력 ---
    # (이전과 동일)
    print("\n--- 추출 완료된 리스트 ---")
    print(f"시각: {times}")
    print(f"날씨: {weathers}")
    print(f"기온(℃): {temperatures}")
    print(f"체감기온(℃): {feels_like_temps}")
    print(f"강수량(mm): {precip_amounts}")
    print(f"강수강도: {precip_intensities}")
    print(f"강수확률(%): {precip_probabilities}")
    print(f"바람방향: {wind_directions}")
    print(f"바람속도(m/s): {wind_speeds}")
    print(f"습도(%): {humidities}")
    print(f"폭염영향: {heatwave_impacts}")

except NoSuchElementException as e:
    print(f"오류: 필수 요소를 찾을 수 없습니다. CSS 선택자를 확인하세요. ({e})")
except Exception as e:
    print(f"예상치 못한 오류 발생: {e}")
    import traceback
    traceback.print_exc()

# finally:
#     # 작업 완료 후 드라이버 종료
#     # driver.quit()

총 15개의 시간대 데이터를 찾았습니다.

--- 추출 완료된 리스트 ---
시각: ['10시', '11시', '12시', '13시', '14시', '15시', '16시', '17시', '18시', '19시', '20시', '21시', '22시', '23시', '0시']
날씨: ['구름 많음', '구름 많음', '구름 많음', '구름 많음', '맑음', '맑음', '맑음', '맑음', '맑음', '맑음', '맑음', '맑음', '맑음', '맑음', '맑음']
기온(℃): [16, 16, 17, 17, 18, 18, 18, 17, 16, 14, 13, 12, 11, 11, 10]
체감기온(℃): [16, 16, 17, 17, 18, 18, 18, 17, 16, 14, 13, 12, 11, 11, 9]
강수량(mm): [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
강수강도: [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
강수확률(%): [None, None, None, None, None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
바람방향: ['서풍', '서풍', '북서풍', '북서풍', '서풍', '서풍', '서풍', '서풍', '서풍', '북서풍', '북서풍', '북서풍', '북서풍', '북서풍', '북서풍']
바람속도(m/s): [1, 2, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 3, 3, 3]
습도(%): [30, 25, 25, 30, 30, 30, 35, 35, 40, 45, 45, 50, 50, 45, 50]
폭염영향: [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]


In [9]:
keys = ['시각', '날씨', '기온(℃)', '체감기온(℃)', '강수량(mm)', '강수강도', '강수확률(%)', '바람방향', '바람속도(m/s)', '습도(%)', '폭염영향']
# 제공된 리스트 변수들을 사용한다고 가정 (times, weathers, temperatures 등)
list_of_lists = [times, weathers, temperatures, feels_like_temps, precip_amounts, precip_intensities, precip_probabilities, wind_directions, wind_speeds, humidities, heatwave_impacts]

structured_data = []
num_items = len(times) # 모든 리스트 길이가 같다고 가정

for i in range(num_items):
    record = {}
    for j, key in enumerate(keys):
         # list_of_lists[j][i] 를 사용하여 올바른 값에 접근
         record[key] = list_of_lists[j][i] 
    structured_data.append(record)

# 이제 structured_data를 JSON으로 변환하여 API에 전달할 수 있습니다.
import json
json_data = json.dumps(structured_data, ensure_ascii=False, indent=2) 
print(type(json_data))
print(json_data)

<class 'str'>
[
  {
    "시각": "10시",
    "날씨": "구름 많음",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 1,
    "습도(%)": 30,
    "폭염영향": null
  },
  {
    "시각": "11시",
    "날씨": "구름 많음",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 25,
    "폭염영향": null
  },
  {
    "시각": "12시",
    "날씨": "구름 많음",
    "기온(℃)": 17,
    "체감기온(℃)": 17,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 25,
    "폭염영향": null
  },
  {
    "시각": "13시",
    "날씨": "구름 많음",
    "기온(℃)": 17,
    "체감기온(℃)": 17,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 30,
    "폭염영향": null
  },
  {
    "시각": "14시",
    "날씨": "맑음",
    "기온(℃)": 18,
    "체감기온(℃)": 18,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": nul

# GEMINI API 연동

In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY 환경 변수를 설정해주세요.")

In [13]:
%pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: c:\Users\dandycode\.pyenv\pyenv-win\versions\3.11.7\python.exe -m pip install --upgrade pip


In [14]:
from google import genai

client = genai.Client(api_key=gemini_api_key)

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Explain how AI works in a few words",
)

print(response.text)

AI learns from data to make predictions or decisions.



In [15]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 날씨 상황을 요약하고, 특히 주목할 만한 변화(예: 강수 시작/종료, 풍속 변화 등)를 설명해주세요.

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 날씨 상황을 요약하고, 특히 주목할 만한 변화(예: 강수 시작/종료, 풍속 변화 등)를 설명해주세요.

**날씨 데이터:**
```json
[
  {
    "시각": "10시",
    "날씨": "구름 많음",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 1,
    "습도(%)": 30,
    "폭염영향": null
  },
  {
    "시각": "11시",
    "날씨": "구름 많음",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 25,
    "폭염영향": null
  },
  {
    "시각": "12시",
    "날씨": "구름 많음",
    "기온(℃)": 17,
    "체감기온(℃)": 17,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 25,
    "폭염영향": null
  },
  {
    "시각": "13시",
    "날씨": "구름 많음",
    "기온(℃)": 17,
    "체감기온(℃)": 17,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 30,
    "폭염영향": null
  },
  {
    "시각": "14시",
    "날씨"

In [16]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
)
print(response.text)

## 날씨 데이터 요약 및 변화 분석

**전반적인 날씨 경향:**

*   **시간대:** 10시부터 익일 0시까지 (총 15시간)
*   **날씨:** 오전에 구름이 많다가 오후부터 밤까지 맑은 날씨가 지속되었습니다.
*   **기온:** 16℃에서 시작하여 18℃까지 상승했다가 점차 하강하여 10℃로 마감되었습니다.
*   **강수:** 강수량은 전 시간대에 걸쳐 없었습니다.
*   **습도:** 습도는 오전 30%에서 시작하여 밤에는 50%까지 증가했습니다.
*   **바람:** 바람은 주로 서풍 또는 북서풍이 불었으며, 풍속은 1~3m/s 정도로 약하게 불었습니다. 폭염 영향은 관측되지 않았습니다.

**주목할 만한 변화:**

*   **날씨 변화:**
    *   **14시:** 구름 많음에서 맑음으로 날씨가 변화했습니다. 이는 가장 뚜렷한 날씨 변화입니다.
*   **기온 변화:**
    *   **10시-14시:** 기온이 16℃에서 18℃로 상승했습니다.
    *   **17시-0시:** 기온이 17℃에서 10℃로 점차적으로 하강했습니다. 야간으로 갈수록 기온이 뚜렷하게 떨어지는 것을 확인할 수 있습니다.
*   **바람 변화:**
    *   **10시-12시:** 바람 방향이 서풍에서 북서풍으로 변경되었습니다.
    *   **11시-12시:** 풍속이 2m/s에서 3m/s로 증가했습니다.
*   **습도 변화:**
    *   **11시-12시:** 습도가 25%에서 25%로 변화가 없습니다.
    *   **18시-21시:** 습도가 40%에서 50%로 증가했습니다. 저녁 시간대에 습도가 높아지는 경향을 보입니다.

**종합 의견:**

오전에는 구름이 많았지만, 시간이 지남에 따라 맑은 날씨로 바뀌었습니다. 기온은 낮 동안 상승했다가 저녁부터 밤까지 점차적으로 하강했습니다. 강수 가능성은 없었으며, 바람은 약하게 불었습니다. 습도는 저녁 시간대에 증가하는 경향을 보였습니다.

이 정보는 특정 지역의 날씨 변화를 이해하고,

In [17]:
import datetime
current_time_local = datetime.datetime.now()
formatted_time= current_time_local.strftime("%Y-%m-%d %H:%M:%S")
formatted_time

'2025-04-25 09:27:41'

In [18]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**오늘 날짜 시간:** {formatted_time}

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**오늘 날짜 시간:** 2025-04-25 09:27:41

**날씨 데이터:**
```json
[
  {
    "시각": "10시",
    "날씨": "구름 많음",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 1,
    "습도(%)": 30,
    "폭염영향": null
  },
  {
    "시각": "11시",
    "날씨": "구름 많음",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 25,
    "폭염영향": null
  },
  {
    "시각": "12시",
    "날씨": "구름 많음",
    "기온(℃)": 17,
    "체감기온(℃)": 17,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 25,
    "폭염영향": null
  },
  {
    "시각": "13시",
    "날씨": "구름 많음",
    "기온(℃)": 17,
    "체감기온(℃)": 17,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 30,
    "폭염영향": null
  },
  {
    "시각": "14시",
   

In [19]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
)
print(response.text)

현재 시간은 2025-04-25 09:27:41 입니다.

JSON 데이터에서 가장 가까운 시간인 10시부터 데이터를 살펴보겠습니다.

*   **강수량(mm)**, **강수강도**, **강수확률(%)** 항목이 모두 `null` 이거나 0입니다.

따라서, 현재 외출 시 우산은 필요하지 않습니다.



In [20]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**오늘 날짜 시간:** {formatted_time}

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**오늘 날짜 시간:** 2025-04-25 09:27:41

**날씨 데이터:**
```json
[
  {
    "시각": "10시",
    "날씨": "구름 많음",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 1,
    "습도(%)": 30,
    "폭염영향": null
  },
  {
    "시각": "11시",
    "날씨": "구름 많음",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 25,
    "폭염영향": null
  },
  {
    "시각": "12시",
    "날씨": "구름 많음",
    "기온(℃)": 17,
    "체감기온(℃)": 17,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 25,
    "폭염영향": null
  },
  {
    "시각": "13시",
    "날씨": "구름 많음",
    "기온(℃)": 17,
    "체감기온(℃)": 17,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 30,
    "폭염영향": null
  },
  {
    "시각"

In [21]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
)
print(response.text)

2025년 4월 25일 09시 27분 현재 날씨 데이터를 기반으로 외출 시 적합한 드레스 코디를 제안합니다.

**날씨 분석:**

*   **기온:** 현재 16도에서 18도까지 오르는 것으로 예상됩니다. 저녁에는 10도까지 떨어집니다.
*   **날씨:** 오전에는 구름이 많지만, 오후에는 맑은 날씨가 예상됩니다.
*   **강수 확률:** 강수 확률은 0%로 비가 올 가능성은 낮습니다.
*   **습도:** 30%에서 50% 정도로 건조한 편입니다.
*   **바람:** 바람은 약하게 불 것으로 예상됩니다.

**추천 드레스 코디:**

*   **기본:**
    *   **상의:** 긴팔 티셔츠 또는 얇은 니트
    *   **하의:** 청바지, 면바지, 또는 스커트
    *   **겉옷:** 가벼운 재킷, 가디건, 또는 트렌치 코트 (저녁에 기온이 떨어지므로 필수)
*   **선택 사항:**
    *   **머플러 또는 스카프:** 바람이 불 때 목을 보호하고 스타일을 더할 수 있습니다.
    *   **선글라스:** 오후에 맑은 날씨가 예상되므로 햇빛을 가리는 데 도움이 됩니다.
    *   **편안한 신발:** 많이 걸을 경우를 대비하여 운동화나 플랫슈즈를 추천합니다.

**종합:**

*   낮에는 활동하기 편한 복장을 하고, 저녁에 쌀쌀해질 수 있으므로 겉옷을 꼭 챙기세요.
*   자외선 차단제와 건조함을 대비하여 보습제를 바르는 것을 추천합니다.
*   변동 가능한 날씨에 대비하여 우산이나 휴대용 우비를 챙기는 것도 좋은 방법입니다.

